<a href="https://colab.research.google.com/github/FranciscoBPereira/AnaliseDados_MEI_2526/blob/main/AD2526_P7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

**Function to generate time series**

In [ ]:
# This function generates multivariate time series
# It generates nr_series series, each one with n_steps values
# Each step has n_features. With multivariate time series (n_features > 1),
# a correlation between them can be specified (default value is 0.7)
# The amount of noise in the series must be specified.

# It returns a NumPy array of shape [nr_series, n_steps, n_features]

def generate_time_series(nr_series, n_steps, n_features, noise_factor, corr_strength=0.7):

    time = np.linspace(0, 1, n_steps)

    freq1, freq2, offset1, offset2 = np.random.rand(4, nr_series, 1)
    base = 0.5 * np.sin((time - offset1) * (freq1 * 10 + 10))
    base += 0.2 * np.sin((time - offset2) * (freq2 * 20 + 20))
    base += noise_factor * (np.random.rand(nr_series, n_steps) - 0.5)

    series = [base]

    for i in range(1, n_features):
        noise = noise_factor * (np.random.rand(nr_series, n_steps) - 0.5)
        correlated = corr_strength * base + (1 - corr_strength) * noise
        series.append(correlated)

    series = np.stack(series, axis=-1)
    return series.astype(np.float32)

In [ ]:
# Function to visualize time series

def plot_multivariate_series(series, y=None, y_pred=None, x_label="t", y_label="x(t)"):

    series = np.asarray(series)

    if series.ndim == 1:
        series = series[:, np.newaxis]

    n_steps, n_features = series.shape
    plt.figure(figsize=(10, 4))

    colors = plt.cm.viridis(np.linspace(0, 1, n_features))

    for i in range(n_features):
        plt.plot(series[:, i], ".-", label=f"Feature {i+1}", color=colors[i])

        if y is not None:
            plt.plot(n_steps, y[i] if np.ndim(y) > 0 else y, "bx", markersize=10)
        if y_pred is not None:
            plt.plot(n_steps, y_pred[i] if np.ndim(y_pred) > 0 else y_pred, "ro")

    plt.grid(True)
    plt.xlabel(x_label, fontsize=16)
    plt.ylabel(y_label, fontsize=16, rotation=0)
    plt.hlines(0, 0, n_steps + 1, linewidth=1, color="gray")

    ymin = series.min() - 0.1
    ymax = series.max() + 0.1
    plt.axis([0, n_steps, ymin, ymax])

    plt.legend()
    plt.show()

**A - Univariate Time Series**

**1. Dataset Creation**

In [ ]:
# Generate the datasets for training, validation and testing

# Each time series has 50 steps

np.random.seed(42)

n_steps = 50
n_features = 1
noise_factor = 0.1

series = generate_time_series(10000, n_steps + 1, n_features, noise_factor)
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:, :n_steps], series[9000:, -1]

print('Training: ', X_train.shape)
print('Validation: ', X_valid.shape)
print('Testing: ', X_test.shape)

In [ ]:
# Plot one example of a time series
# Change the value of index to check another example


index = 1

plot_multivariate_series((X_train[index]), y_train[index])



**2. Baseline Approaches**

In [ ]:
# Baseline 1

# The predictor returns the last value of the series
# We don't even need a model for this baseline

# Baseline 1 is evaluated in the test dataset

from tensorflow.keras.losses import MeanSquaredError

mse = MeanSquaredError()

y_pred = X_test[:, -1]

difference = np.absolute(y_pred - y_test)
print('Min: ', np.min(difference))
print('Max: ', np.max(difference))

# MSE Test error

print('MSE: ', np.mean(mse(y_test, y_pred)))


In [ ]:
# Plot the error in the first test timeseries
# The red circle identifies the prediction and the X mark the correct value

# Try other timeseries
serie = 0

plot_multivariate_series(X_test[serie, :, 0], y_test[serie, 0], y_pred[serie, 0])
plt.show()

In [ ]:
# Baseline 2

# Rely on a simple fully connected network the linearly combines inputs to predict the next value
# Training finds the best weights of the linear combination

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

# The Flatten layer linearizes inputs (the 5o values of the timeseries)
# The output is the next predicted value

modelB2 = keras.models.Sequential([
    layers.Input(shape=[50, 1]),
    layers.Flatten(),
    layers.Dense(1)
])

modelB2.summary()


In [ ]:
modelB2.compile(loss='mse', optimizer='adam')

history = modelB2.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))

In [ ]:
# MSE test error

modelB2.evaluate(X_test, y_test)

In [ ]:
# Plot the error in the first test timeseries
# The red circle identifies the prediction and the X mark the correct value

# Try other timeseries

serie = 0

y_pred = modelB2.predict(X_valid)
plot_multivariate_series(X_valid[serie, :, 0], y_valid[serie, 0], y_pred[serie, 0])
plt.show()

In [ ]:
# Collect the absolute values of the weights of the neural network (at the end of training)

weights, bias = modelB2.layers[1].get_weights()
weights = np.absolute(weights)

# Plot the weight values

plt.plot(weights)

**Quiz:**

1. Compare the performace of the 2 baselines

2. Analyze the weight values that appear in the plot. How do you interpret the pattern of the line?

**3. Recurrent Approaches**

In [ ]:
# We already have MSE values obtained with two simple baseline approaches

# Now we can move to recurrent networks and check how they can improve performance

# https://www.tensorflow.org/guide/keras/rnn
# https://www.tensorflow.org/api_docs/python/tf/keras/layers/SimpleRNN

# RNN 1

# It relies on a single RNN cell that processes input sequentialy and outputs a prediction

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)


# We don't have to specify the length of the input

modelR1 = keras.models.Sequential([
    layers.Input(shape=[None, 1]),
    layers.SimpleRNN(1)
])

modelR1.summary()

In [ ]:

modelR1.compile(loss='mse', optimizer='adam')

history = modelR1.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))

In [ ]:
# Evaluate MSE on test set

modelR1.evaluate(X_test, y_test)

In [ ]:
#RNN 2

# Add additional RNN cells, aiming at enhancing performance (Deep RNN)
# Wou have a budget of 12 RNN cells and can choose how to organize them

# When adopting a multi-layered approach, recall that the Boolean parameter return_sequences
# indicates the type of output of a recurring layer. When set to true, the entire sequence is returned.
# If false (default value), it only returns the last result

# A final Dense layer with just one cell to output the prediction (a single value, do we don't need recurrent cells here)

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)


modelR2 = keras.models.Sequential([
    layers.Input(shape=[None, 1]),

    ## COMPLETE ###

    layers.Dense(1)
])

modelR2.summary()

In [ ]:
modelR2.compile(loss='mse', optimizer='adam')

history = modelR2.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))

In [ ]:
# Evalue MSE on test set

modelR2.evaluate(X_test, y_test)

In [ ]:
#RNN 3

# Replace RNN cells by LSTM or GRU
# Wou have the same budget of 12 cells and can choose how to organize them

# A final Dense layer with just one cell to output the prediction (a single value, do we don't need recurrent cells here)

keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)


modelR3 = keras.models.Sequential([
    layers.Input(shape=[None, 1]),

    ### COMPLETE ###

    layers.Dense(1)
])

modelR3.summary()

In [ ]:
modelR3.compile(loss='mse', optimizer='adam')

history = modelR3.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))

In [ ]:
# Evalue MSE on test set

modelR3.evaluate(X_test, y_test)

**B - Multivariate Time Series**

In [ ]:
# Generate the datasets for training, validation and testing

# Each time series has 50 steps

np.random.seed(42)

n_steps = 50
n_features = 2
noise_factor = 0.25
correlation = 0.75

series = generate_time_series(10000, n_steps + 1, n_features, noise_factor, correlation)
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:, :n_steps], series[9000:, -1]

print('Training: ', X_train.shape)
print('Validation: ', X_valid.shape)
print('Testing: ', X_test.shape)

In [ ]:
# Plot one example of a time series
# Change the value of index to check another example

index = 0

plot_multivariate_series((X_train[index]), y_train[index])

In [ ]:

### Test the 5 approaches (2 baseline + 3 recurrent) with this new example

### Code goes here ###



